# Short notebook 2 — long-format XGBoost (Kaggle public 0.98058)

This notebook reproduces our first Kaggle submission (public leaderboard **0.98058**) from the raw
competition data. It is the end point of the feature-engineering track described in the report
(logs 001–036): one row per (case, hour, unit), a single binary XGBoost model shared by all units,
207 features.

1. **Base table** — join price, inflow, storage and water value onto one row per (case, hour).
2. **Feature stages** — economic spreads and margins, terminal water value (read at start + 7
   days), seasonal percentiles, water budget, merit order, cascade graph, calendar.
3. **Count table** — one row per (case, unit) with multi-horizon storage/inflow/production
   features; a 6-class XGBoost model over hours-on bins is fitted **only to rank** these
   features, and the top 120 (+ up to 30 unit-relative) are attached per unit.
4. **Hourly structure** — block/run features within the week.
5. **Long format + XGBoost** — 600 rounds, depth 8, `min_child_weight=500`, on all 2,922
   labelled cases; predict the 725 test cases.

**Resources.** Full resolution trains on 6.87M rows × ~207 features. On a 4-core machine it
takes about 40 minutes and peaks at **~24 GB of memory** (RAM plus swap), once while XGBoost
quantises the training matrix and again at prediction. A 16 GB machine needs swap; macOS adds it
automatically. `STRIDE = 3` trains on every
third case for a quicker, lighter run.

**Requirements:** the versions the uploaded file was produced with: `numpy==2.2.6 pandas==2.3.2
pyarrow==23.0.1 xgboost==3.4.1 scikit-learn==1.7.2 PyYAML==6.0.3`.

**Reproducibility.** The uploaded file was produced on an Apple Silicon Mac. On a 4-core x86 Linux
machine with the same package versions this notebook selects 143 count-table features instead of
141, so the final model differs slightly: its predictions correlate 0.995 with the uploaded file
and fall on the same side of 0.5 in 98% of cells. The difference is not the thread count (4 and 8
threads select identical features); we attribute it to platform-level floating-point differences in
XGBoost's histogram sums, which reorder near-tied features in the count model's importance ranking.
The last cell compares the result with the uploaded file when `submission.csv` is present.

In [1]:
from __future__ import annotations

import gc
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")

# Folder holding data/kernel, data/extended, prediction_mapping.csv, sample_submission.csv
CANDIDATES = [Path("."), Path(".."), Path("sintef_project"), Path("../sintef_project")]
ROOT = next(p.resolve() for p in CANDIDATES if (p / "data/kernel").is_dir())
_ROOT = ROOT
STRIDE = 1                                    # 1 = full resolution (the submitted model)
OUT_PATH = Path("submission_xgboost.csv")
print("data root:", ROOT)

data root: /home/user/ML_Project/sintef_project


## 1. Base table: one row per (case, hour)

In [2]:
N_HOURS = 168


def build_base_tables():
    """Raw CSVs -> one row per (case, hour): joined inputs, targets for train cases."""
    price = pd.read_csv(ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv")
    inflow = pd.read_csv(ROOT / "data/kernel/Historical_inflow_1958_2025.csv")
    volume = pd.read_csv(ROOT / "data/kernel/Historical_volume_2015_2024.csv")
    water_value = pd.read_csv(ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv")
    uc = pd.read_csv(ROOT / "data/kernel/Unit_commitment_decisions.csv")
    pred_mapping = pd.read_csv(ROOT / "prediction_mapping.csv")

    for df in (price, inflow, volume, water_value):
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    uc["starttime"] = pd.to_datetime(uc["starttime"])
    pred_mapping["starttime"] = pd.to_datetime(pred_mapping["starttime"])

    price = price.rename(columns={"date": "timestamp", "Day-ahead price (EUR/MWh)": "price_eur_mwh"})
    inflow = inflow.rename(columns={"date": "timestamp"})
    inflow = inflow.rename(columns={c: f"inflow_{sanitize(c)}" for c in inflow.columns if c != "timestamp"})
    volume = volume.rename(columns={"date": "day"})
    volume = volume.rename(columns={c: f"storage_{sanitize(c)}" for c in volume.columns if c != "day"})
    water_value = water_value.rename(columns={"date": "day"})
    water_value = water_value.rename(columns={c: f"water_value_{sanitize(c)}" for c in water_value.columns if c != "day"})

    # --- spine: one row per (case, hour) for both train and test cases ---
    cases = pd.concat([
        uc[["Run No", "starttime"]].assign(split="train"),
        pred_mapping[["Run No", "starttime"]].assign(split="test"),
    ], ignore_index=True).rename(columns={"Run No": "run_no"})

    n_cases = len(cases)
    run_no_rep = np.repeat(cases["run_no"].values, N_HOURS)
    starttime_rep = np.repeat(cases["starttime"].values, N_HOURS)
    split_rep = np.repeat(cases["split"].values, N_HOURS)
    hour_rep = np.tile(np.arange(N_HOURS), n_cases)
    timestamp = starttime_rep + hour_rep * np.timedelta64(1, "h")

    long_df = pd.DataFrame({
        "run_no": run_no_rep,
        "split": split_rep,
        "starttime": starttime_rep,
        "hour": hour_rep,
        "timestamp": timestamp,
    })
    long_df["day"] = long_df["timestamp"].dt.floor("D")

    # --- join raw features ---
    long_df = long_df.merge(price, on="timestamp", how="left")
    long_df = long_df.merge(inflow, on="timestamp", how="left")
    long_df = long_df.merge(volume, on="day", how="left")
    long_df = long_df.merge(water_value, on="day", how="left")

    missing = long_df.filter(regex="^(price_|inflow_|storage_|water_value_)").isna().sum()
    missing = missing[missing > 0]
    if len(missing):
        print("WARNING - unmatched rows after join:")
        print(missing)

    # --- reshape targets (train only) from wide to long ---
    label_cols = [c for c in uc.columns if c.startswith("result_committed_")]
    units = sorted({re.match(r"result_committed_(.+)_t\d+$", c).group(1) for c in label_cols})

    target_frames = []
    uc_indexed = uc.set_index("Run No")
    for u in units:
        cols = [f"result_committed_{u}_t{t}" for t in range(N_HOURS)]
        sub = uc_indexed[cols].copy()
        sub.columns = pd.RangeIndex(N_HOURS, name="hour")
        stacked = sub.stack()
        stacked.name = f"on_{sanitize(u)}"
        target_frames.append(stacked)
    target_long = pd.concat(target_frames, axis=1).reset_index().rename(columns={"Run No": "run_no"})
    target_cols = [c for c in target_long.columns if c.startswith("on_")]

    id_cols = ["run_no", "split", "starttime", "timestamp", "hour"]
    feature_cols = [c for c in long_df.columns if c not in id_cols + ["day"]]

    train_df = long_df[long_df["split"] == "train"].merge(target_long, on=["run_no", "hour"], how="left")
    train_df = train_df[id_cols + feature_cols + target_cols]

    test_df = long_df[long_df["split"] == "test"][id_cols + feature_cols]

    assert train_df[target_cols].isna().sum().sum() == 0, "unexpected missing targets in train"
    assert len(train_df) == len(uc) * N_HOURS
    assert len(test_df) == len(pred_mapping) * N_HOURS

    # the pipeline was developed on these tables read back from parquet (nanosecond
    # timestamps); pandas >= 3 builds microseconds, so restore the original resolution
    for d in (train_df, test_df):
        for c in ("starttime", "timestamp"):
            d[c] = d[c].astype("datetime64[ns]")
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


## 2. Feature stages over the (case, hour) table

Built on train and test together so that seasonal and relative features share one scale. The
15 pruned economic features were selected by gain from 299 candidates in log 015.

In [3]:
PRUNED_FEATURES = (
    "f15_units_justified_Kjela",
    "f14_upstream_spread_Byrte",
    "f15_cap_share_needed_Kjela",
    "f15_cap_share_needed_Byrte",
    "f07_price_minus_terminal_wv",
    "f15_units_justified_Vinje",
    "f06_spread_sys",
    "f15_units_justified_Vesle_Kjela",
    "f15_cap_share_needed_Vesle_Kjela",
    "f06x_margin_Haukeli",
    "f06_spread_Byrte",
    "f10_breakeven_hours_Kjela",
    "f14_upstream_spread_Songa",
    "f06_spread_Vesle_Kjela",
    "f05_wv_dev_Kjela",
)


TOPOLOGY = _ROOT / "data/extended/Tokke_Vinje_topology.yaml"


_topo_cache = None


def load_topology() -> dict:
    global _topo_cache
    if _topo_cache is None:
        with open(TOPOLOGY) as f:
            _topo_cache = yaml.safe_load(f)
    return _topo_cache


def sanitize(name: str) -> str:
    return name.replace(" ", "_")


def stage_calendar(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 17/18: calendar position. Hour-of-day is hourly-varying; the rest are case-level."""
    ts = df["timestamp"]
    out = pd.DataFrame(index=df.index)
    out["f17_hour_of_day"] = ts.dt.hour
    out["f17_day_of_week"] = ts.dt.dayofweek
    out["f17_is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
    out["f17_day_of_year"] = ts.dt.dayofyear
    out["f17_month"] = ts.dt.month
    out["f17_doy_sin"] = np.sin(2 * np.pi * ts.dt.dayofyear / 365.25)
    out["f17_doy_cos"] = np.cos(2 * np.pi * ts.dt.dayofyear / 365.25)
    out["f18_day_in_case"] = df["hour"] // 24
    out["f18_hours_remaining"] = 167 - df["hour"]
    return out


def stage_graph_cascade(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 14: upstream cascade aggregates via multi-hop traversal of the topology graph."""
    topo = load_topology()
    conns = topo["connections"]
    upstream: dict[str, set] = {}

    def walk(node, seen=None):
        seen = seen or set()
        acc = set()
        for c in conns:
            if c["to"] == node and c["from"] not in seen:
                seen.add(c["from"])
                if c["from_type"] == "reservoir":
                    acc.add(c["from"])
                acc |= walk(c["from"], seen)
        return acc

    plants = list(topo["model"]["plant"])
    for p in plants:
        upstream[p] = walk(p)

    out = pd.DataFrame(index=df.index)
    for plant, ups in upstream.items():
        pl = sanitize(plant)
        scols = [f"storage_{sanitize(r)}" for r in ups if f"storage_{sanitize(r)}" in df.columns]
        wcols = [f"water_value_{sanitize(r)}" for r in ups if f"water_value_{sanitize(r)}" in df.columns]
        icols = [f"inflow_{sanitize(r)}" for r in ups if f"inflow_{sanitize(r)}" in df.columns]
        if scols:
            out[f"f14_upstream_storage_{pl}"] = df[scols].sum(axis=1)
        if wcols:
            out[f"f14_upstream_wv_{pl}"] = df[wcols].mean(axis=1)
            out[f"f14_upstream_spread_{pl}"] = df["price_eur_mwh"] - df[wcols].mean(axis=1)
        if icols:
            out[f"f14_upstream_inflow_{pl}"] = df[icols].sum(axis=1)
        out[f"f14_n_upstream_{pl}"] = len(ups)
    return out


def expanding_trailing_mean(series: pd.Series, at_dates: pd.Series, days: int) -> np.ndarray:
    """Mean of the series over the `days` before each target date (strictly backward-looking)."""
    s = series.sort_index().dropna()
    roll = s.rolling(f"{days}D", min_periods=2).mean()
    return roll.reindex(pd.to_datetime(pd.Series(at_dates)), method="ffill").to_numpy()


def stage_inflow_regime(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 9: wet-year / dry-year regime (+0.331 for the 365d trailing mean)."""
    inf = pd.read_csv(_ROOT / "data/kernel/Historical_inflow_1958_2025.csv")
    inf["date"] = pd.to_datetime(inf["date"]).dt.tz_localize(None)
    inf_d = inf.set_index("date").resample("D").mean()
    vol_cols = pd.read_csv(_ROOT / "data/kernel/Historical_volume_2015_2024.csv", nrows=1).columns
    res = [c for c in inf_d.columns if c in set(vol_cols)]
    sys_inflow = inf_d[res].sum(axis=1)

    case_day = df.groupby("run_no")["starttime"].first()
    feats = pd.DataFrame(index=case_day.index)
    for w in (30, 90, 365):
        feats[f"f09_inflow_trail_{w}d"] = expanding_trailing_mean(sys_inflow, case_day, w)
    feats["f09_inflow_30_over_365"] = (
        feats["f09_inflow_trail_30d"] / feats["f09_inflow_trail_365d"].replace(0, np.nan)
    )
    return df[["run_no"]].merge(feats, left_on="run_no", right_index=True, how="left") \
                          .drop(columns="run_no").set_index(df.index)


def plant_to_reservoir() -> dict[str, str]:
    """Direct reservoir -> plant edges. Songa has none (it is fed via two tunnels)."""
    topo = load_topology()
    return {c["to"]: c["from"] for c in topo["connections"]
            if c["to_type"] == "plant" and c["from_type"] == "reservoir"}


def stage_merit_order(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 15: plant capacity structure crossed with the hourly economic signal.

    Merit rank itself is static per unit and therefore invisible in this row layout, so it only
    enters combined with hourly quantities.
    """
    topo = load_topology()
    gens = topo["model"]["generator"]
    p2r = plant_to_reservoir()
    wv_cols = [c for c in df.columns if c.startswith("water_value_")]
    sys_wv = df[wv_cols].mean(axis=1)
    out = pd.DataFrame(index=df.index)
    for plant, r in p2r.items():
        pl = sanitize(plant)
        pgens = sorted(g for g in gens if sanitize(g).startswith(pl + "_G"))
        if not pgens:
            continue
        col = f"water_value_{sanitize(r)}"
        sp = df["price_eur_mwh"] - (df[col] if col in df.columns else sys_wv)
        cap_each = gens[pgens[0]]["p_max"]
        p_min = gens[pgens[0]]["p_min"]
        # how many units the hourly margin can justify running, given each unit's minimum output
        out[f"f15_units_justified_{pl}"] = (sp.clip(lower=0) * cap_each / max(p_min, 1e-6)).clip(upper=len(pgens))
        out[f"f15_cap_share_needed_{pl}"] = sp.clip(lower=0) / max(sp.abs().max(), 1e-6)
        out[f"f15_n_units_{pl}"] = len(pgens)
    return out


def stage_physics(df: pd.DataFrame) -> pd.DataFrame:
    """Themes 10-12: head, energy conversion, start-cost economics, penstock structure.

    Static per-unit constants are useless in this row layout, so everything here is either a
    per-plant time-varying quantity or a combination of a static attribute with one.
    """
    topo = load_topology()
    gens, plants, reservoirs = topo["model"]["generator"], topo["model"]["plant"], topo["model"]["reservoir"]
    p2r = plant_to_reservoir()
    out = pd.DataFrame(index=df.index)

    for plant, r in p2r.items():
        scol = f"storage_{sanitize(r)}"
        if scol not in df.columns:
            continue
        spec = reservoirs[r]
        vh = spec["vol_head"]
        # storage -> reservoir level -> gross head above the plant's outlet
        level = np.interp(df[scol].to_numpy(), vh["x"], vh["y"])
        head = level - plants[plant]["outlet_line"]
        pl = sanitize(plant)
        out[f"f11_head_{pl}"] = head
        lo, hi = min(vh["y"]) - plants[plant]["outlet_line"], max(vh["y"]) - plants[plant]["outlet_line"]
        out[f"f11_head_frac_{pl}"] = (head - lo) / (hi - lo) if hi > lo else 0.0

        pgens = [g for g in gens if sanitize(g).startswith(pl + "_G")]
        if not pgens:
            continue
        g0 = gens[pgens[0]]
        # turbine efficiency interpolated across the head-referenced curves
        refs = np.array([c["ref"] for c in g0.get("turb_eff_curves", [])], dtype=float)
        peaks = np.array([max(c["y"]) for c in g0.get("turb_eff_curves", [])], dtype=float)
        if len(refs) >= 2:
            order = np.argsort(refs)
            out[f"f11_turb_eff_{pl}"] = np.interp(head, refs[order], peaks[order])
        else:
            out[f"f11_turb_eff_{pl}"] = peaks[0] if len(peaks) else np.nan
        # MWh per m3: rho*g*H*eta / 3.6e9, with eta as a fraction
        out[f"f11_mwh_per_m3_{pl}"] = (
            1000 * 9.81 * head * (out[f"f11_turb_eff_{pl}"] / 100.0) / 3.6e9
        )

        # start-cost economics: break-even run length, using the plant's own spread
        spread_col = f"f06_spread_{pl}"
        spread = df[spread_col] if spread_col in df.columns else None
        if spread is None:
            wvcol = f"water_value_{sanitize(r)}"
            spread = df["price_eur_mwh"] - df[wvcol] if wvcol in df.columns else None
        if spread is not None:
            p_max_total = sum(gens[g]["p_max"] for g in pgens)
            startcost = g0.get("startcost_const", np.nan)
            hourly_margin = spread * gens[pgens[0]]["p_max"]
            out[f"f10_breakeven_hours_{pl}"] = startcost / hourly_margin.where(hourly_margin > 0)
            out[f"f10_startcost_mwh_equiv_{pl}"] = startcost / df["price_eur_mwh"].clip(lower=0.1)
            out[f"f11_plant_energy_capacity_{pl}"] = p_max_total * out[f"f11_head_frac_{pl}"]

        # penstock structure: how many distinct penstocks the plant's units sit on
        pens = {gens[g].get("penstock", 1) for g in pgens}
        out[f"f12_n_penstocks_{pl}"] = len(pens)
        out[f"f12_units_per_penstock_{pl}"] = len(pgens) / max(len(pens), 1)

    return out


def stage_price_relative(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 2: case-relative, scale-free price (+0.263 z-score, +0.252 percentile)."""
    g = df.groupby("run_no")["price_eur_mwh"]
    mean, std = g.transform("mean"), g.transform("std")
    p = df["price_eur_mwh"]
    out = pd.DataFrame(index=df.index)
    out["f02_price_z_case"] = (p - mean) / std.replace(0, np.nan)
    out["f02_price_pct_case"] = g.rank(pct=True)
    out["f02_price_log_ratio"] = np.log(p.clip(lower=0.1)) - np.log(mean.clip(lower=0.1))
    out["f02_price_case_std"] = std
    day = df["timestamp"].dt.floor("D")
    gd = df.groupby([df["run_no"], day])["price_eur_mwh"]
    out["f02_price_z_day"] = (p - gd.transform("mean")) / gd.transform("std").replace(0, np.nan)
    out["f02_price_rank_in_day"] = gd.rank()
    out["f02_price_minus_case_min"] = p - g.transform("min")
    out["f02_case_max_minus_price"] = g.transform("max") - p
    return out


def stage_price_shape(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 4: where this hour sits in the week's price structure. All hourly-varying."""
    g = df.groupby("run_no")["price_eur_mwh"]
    rank_desc = g.rank(ascending=False, method="first")
    out = pd.DataFrame(index=df.index)
    for k in (12, 24, 48, 72):
        out[f"f04_in_top_{k}h"] = (rank_desc <= k).astype(int)
    out["f04_price_cummean_case"] = g.cummean() if hasattr(g, "cummean") else \
        g.transform(lambda s: s.expanding().mean())
    piv = df.pivot(index="run_no", columns="hour", values="price_eur_mwh").sort_index()
    P = piv.to_numpy()
    idx = pd.MultiIndex.from_product([piv.index, piv.columns], names=["run_no", "hour"])
    target = pd.MultiIndex.from_arrays([df["run_no"], df["hour"]])
    # signed distance to the week's cheapest / dearest hour
    argmax, argmin = P.argmax(axis=1), P.argmin(axis=1)
    hours = np.arange(168)[None, :]
    out["f04_hours_to_case_peak"] = pd.Series((argmax[:, None] - hours).ravel(), index=idx).reindex(target).to_numpy()
    out["f04_hours_since_case_trough"] = pd.Series((hours - argmin[:, None]).ravel(), index=idx).reindex(target).to_numpy()
    # share of the week's total price mass consumed so far
    csum = np.cumsum(P, axis=1)
    out["f04_cum_price_share"] = pd.Series((csum / csum[:, -1:]).ravel(), index=idx).reindex(target).to_numpy()
    return out


def stage_price_windows(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 3: forward/backward price windows, clipped to the case horizon."""
    piv = df.pivot(index="run_no", columns="hour", values="price_eur_mwh").sort_index()
    P = piv.to_numpy()
    csum = np.cumsum(np.concatenate([np.zeros((P.shape[0], 1)), P], axis=1), axis=1)
    idx = pd.MultiIndex.from_product([piv.index, piv.columns], names=["run_no", "hour"])
    target = pd.MultiIndex.from_arrays([df["run_no"], df["hour"]])
    out = pd.DataFrame(index=df.index)
    for w in (6, 12, 24, 48, 72):
        hi = np.minimum(np.arange(168) + w, 168); lo = np.arange(168)
        fwd = (csum[:, hi] - csum[:, lo]) / np.maximum(hi - lo, 1)
        lo2 = np.maximum(np.arange(168) - w + 1, 0); hi2 = np.arange(168) + 1
        bwd = (csum[:, hi2] - csum[:, lo2]) / np.maximum(hi2 - lo2, 1)
        out[f"f03_price_fwd_{w}h"] = pd.Series(fwd.ravel(), index=idx).reindex(target).to_numpy()
        out[f"f03_price_minus_fwd_{w}h"] = pd.Series((P - fwd).ravel(), index=idx).reindex(target).to_numpy()
        out[f"f03_price_minus_bwd_{w}h"] = pd.Series((P - bwd).ravel(), index=idx).reindex(target).to_numpy()
    return out


def expanding_seasonal_percentile(series: pd.Series, at_dates: pd.Series,
                                   window_days: int = 7) -> np.ndarray:
    """Percentile of each value against the same time of year in *strictly earlier* years.

    For each target date d, compares series[d] against all historical observations whose
    day-of-year is within +/- `window_days` of d AND whose date is before d. Returns NaN where
    fewer than 5 prior observations exist (the first year or so).

    The expanding restriction is what makes this safe: a 2019 case is never ranked against 2020+.
    """
    s = series.sort_index().dropna()
    idx_doy = s.index.dayofyear.to_numpy()
    idx_time = s.index.to_numpy()
    vals = s.to_numpy()

    out = np.full(len(at_dates), np.nan)
    for i, d in enumerate(pd.to_datetime(pd.Series(at_dates)).to_numpy()):
        d_ts = pd.Timestamp(d)
        doy = d_ts.dayofyear
        # circular day-of-year distance
        delta = np.abs(idx_doy - doy)
        delta = np.minimum(delta, 365 - delta)
        mask = (delta <= window_days) & (idx_time < d)
        if mask.sum() < 5:
            continue
        prior = vals[mask]
        cur = vals[idx_time == d]
        if len(cur) == 0:
            # value not in the series itself; fall back to the nearest earlier observation
            earlier = vals[idx_time < d]
            if len(earlier) == 0:
                continue
            cur = earlier[-1:]
        out[i] = (prior < cur[0]).mean()
    return out


def stage_seasonal(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 1: seasonal normalisation of the slow series (strongest EDA evidence, +0.537)."""
    kernel = _ROOT / "data/kernel"
    vol = pd.read_csv(kernel / "Historical_volume_2015_2024.csv")
    wv = pd.read_csv(kernel / "Synthetic_water_value_2015_2024.csv")
    inf = pd.read_csv(kernel / "Historical_inflow_1958_2025.csv")
    for d in (vol, wv, inf):
        d["date"] = pd.to_datetime(d["date"]).dt.tz_localize(None)
    vol, wv = vol.set_index("date"), wv.set_index("date")
    inf_d = inf.set_index("date").resample("D").mean()

    res = list(vol.columns)
    sys_storage = vol[res].sum(axis=1)
    sys_wv = wv[res].mean(axis=1)
    sys_inflow = inf_d[[c for c in inf_d.columns if c in res]].sum(axis=1)

    case_day = df.groupby("run_no")["starttime"].first()
    feats = pd.DataFrame(index=case_day.index)
    feats["f01_storage_pct_doy"] = expanding_seasonal_percentile(sys_storage, case_day)
    feats["f01_wv_pct_doy"] = expanding_seasonal_percentile(sys_wv, case_day)
    feats["f01_inflow_pct_doy"] = expanding_seasonal_percentile(sys_inflow, case_day)
    # absolute anomaly as well as rank - trees may use them differently
    feats["f01_storage_anom"] = (
        sys_storage.reindex(case_day.values).to_numpy()
        - expanding_trailing_mean(sys_storage, case_day, 365)
    )
    return df[["run_no"]].merge(feats, left_on="run_no", right_index=True, how="left") \
                          .drop(columns="run_no").set_index(df.index)


def global_seasonal_percentile(series: pd.Series, at_dates: pd.Series,
                                window_days: int = 7) -> np.ndarray:
    """Seasonal percentile against the WHOLE history, including later years.

    The competition withholds only test-period targets, so feature-side statistics computed over
    the full dataset are permitted. This stays valid for CV because the reference distribution is
    identical for train and test rows. It is strictly more informative than the expanding version
    in early years, where little prior history exists.
    """
    s = series.sort_index().dropna()
    idx_doy = s.index.dayofyear.to_numpy()
    vals = s.to_numpy()
    idx_time = s.index.to_numpy()

    out = np.full(len(at_dates), np.nan)
    for i, d in enumerate(pd.to_datetime(pd.Series(at_dates)).to_numpy()):
        doy = pd.Timestamp(d).dayofyear
        delta = np.abs(idx_doy - doy)
        delta = np.minimum(delta, 365 - delta)
        prior = vals[delta <= window_days]
        if len(prior) < 5:
            continue
        cur = vals[idx_time == d]
        if len(cur) == 0:
            earlier = vals[idx_time < d]
            if len(earlier) == 0:
                continue
            cur = earlier[-1:]
        out[i] = (prior < cur[0]).mean()
    return out


def stage_seasonal_global(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 1, global variant. A/B against stage_seasonal (expanding)."""
    kernel = _ROOT / "data/kernel"
    vol = pd.read_csv(kernel / "Historical_volume_2015_2024.csv")
    wv = pd.read_csv(kernel / "Synthetic_water_value_2015_2024.csv")
    inf = pd.read_csv(kernel / "Historical_inflow_1958_2025.csv")
    for d in (vol, wv, inf):
        d["date"] = pd.to_datetime(d["date"]).dt.tz_localize(None)
    vol, wv = vol.set_index("date"), wv.set_index("date")
    inf_d = inf.set_index("date").resample("D").mean()
    res = list(vol.columns)

    case_day = df.groupby("run_no")["starttime"].first()
    feats = pd.DataFrame(index=case_day.index)
    feats["f01g_storage_pct_doy"] = global_seasonal_percentile(vol[res].sum(axis=1), case_day)
    feats["f01g_wv_pct_doy"] = global_seasonal_percentile(wv[res].mean(axis=1), case_day)
    feats["f01g_inflow_pct_doy"] = global_seasonal_percentile(
        inf_d[[c for c in inf_d.columns if c in res]].sum(axis=1), case_day)
    return df[["run_no"]].merge(feats, left_on="run_no", right_index=True, how="left") \
                          .drop(columns="run_no").set_index(df.index)


def stage_spread(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 6: price - water value spread. Raw, NOT normalised (+0.241 vs +0.114 z-scored)."""
    wv_cols = [c for c in df.columns if c.startswith("water_value_")]
    sys_wv = df[wv_cols].mean(axis=1)
    p2r = plant_to_reservoir()
    out = pd.DataFrame(index=df.index)
    out["f06_spread_sys"] = df["price_eur_mwh"] - sys_wv
    for plant, r in p2r.items():
        col = f"water_value_{sanitize(r)}"
        if col in df.columns:
            out[f"f06_spread_{sanitize(plant)}"] = df["price_eur_mwh"] - df[col]
    out["f06_spread_sys_positive"] = (out["f06_spread_sys"] > 0).astype(int)
    out["f06_spread_hours_positive_case"] = (
        out.groupby(df["run_no"])["f06_spread_sys_positive"].transform("sum")
    )
    return out


def stage_spread_extended(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 6 extensions. The spread family supplied 60.8% of model gain, so push on it."""
    p2r = plant_to_reservoir()
    gens = load_topology()["model"]["generator"]
    wv_cols = [c for c in df.columns if c.startswith("water_value_")]
    sys_wv = df[wv_cols].mean(axis=1)
    g = df.groupby("run_no")
    out = pd.DataFrame(index=df.index)

    base = df["price_eur_mwh"] - sys_wv
    out["f06x_spread_rank_case"] = base.groupby(df["run_no"]).rank(pct=True)
    out["f06x_spread_cummax_case"] = base.groupby(df["run_no"]).cummax()
    out["f06x_spread_vs_case_max"] = base.groupby(df["run_no"]).transform("max") - base
    out["f06x_spread_vs_case_mean"] = base - base.groupby(df["run_no"]).transform("mean")

    # forward-looking spread within the horizon: is a profitable block coming?
    piv = df.pivot(index="run_no", columns="hour", values="price_eur_mwh").sort_index()
    wv_piv = sys_wv.groupby([df["run_no"], df["hour"]]).first().unstack().sort_index()
    S = (piv - wv_piv).to_numpy()
    csum = np.cumsum(np.concatenate([np.zeros((S.shape[0], 1)), S], axis=1), axis=1)
    pos = np.cumsum(np.concatenate([np.zeros((S.shape[0], 1)), (S > 0).astype(float)], axis=1), axis=1)
    idx = pd.MultiIndex.from_product([piv.index, piv.columns], names=["run_no", "hour"])
    for w in (12, 24, 48):
        hi = np.minimum(np.arange(168) + w, 168)
        lo = np.arange(168)
        fwd = (csum[:, hi] - csum[:, lo]) / np.maximum(hi - lo, 1)
        fwd_pos = (pos[:, hi] - pos[:, lo])
        out[f"f06x_spread_fwd_{w}h"] = pd.Series(fwd.ravel(), index=idx).reindex(
            pd.MultiIndex.from_arrays([df["run_no"], df["hour"]])).to_numpy()
        out[f"f06x_profitable_hours_next_{w}h"] = pd.Series(fwd_pos.ravel(), index=idx).reindex(
            pd.MultiIndex.from_arrays([df["run_no"], df["hour"]])).to_numpy()

    # per-plant economic margin, scaled by that plant's capacity
    for plant, r in p2r.items():
        pl, col = sanitize(plant), f"water_value_{sanitize(r)}"
        if col not in df.columns:
            continue
        pgens = [g_ for g_ in gens if sanitize(g_).startswith(pl + "_G")]
        cap = sum(gens[g_]["p_max"] for g_ in pgens) or np.nan
        sp = df["price_eur_mwh"] - df[col]
        out[f"f06x_margin_{pl}"] = sp * cap
        out[f"f06x_spread_pos_{pl}"] = (sp > 0).astype(int)
    return out


def stage_terminal_wv(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 7: terminal water value -- the boundary condition of SHOP's own objective.

    `Dataset_definitions_and_explanation.pdf`, under "Reservoir water value", specifies the
    scheduling problem exactly: for a case starting 2015-01-01 00:00 and ending 2015-01-07 23:00,
    the initial volume is the value at 2015-01-01 00:00 and "its final water value should be the
    value at 2015-01-08 00:00" -- i.e. **start + 7 days**.

    This previously read `days=8`, so every case priced its terminal water one day late. The
    quantity matters: SHOP maximises revenue plus the salvage value of water left in the
    reservoir, so the terminal water value IS the marginal cost of generating. A unit runs when
    price exceeds it. That makes `price - terminal_wv` the economically correct spread, and the
    contemporaneous daily water value -- which `f06_spread_*` uses -- the wrong quantity: the
    daily series is an input to the problem, not the price its decisions are taken against.

    Per-reservoir terminal values are emitted alongside the system mean, so each plant can be
    priced against its own reservoir rather than a basin-wide average.

    Verified unavailable for exactly one test case (2024-12-25); falls back to the last
    observation, which is recorded rather than silently NaN.
    """
    wv = pd.read_csv(_ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv")
    wv["date"] = pd.to_datetime(wv["date"]).dt.tz_localize(None)
    wv = wv.set_index("date").sort_index()
    res = list(wv.columns)
    sys_wv = wv[res].mean(axis=1)

    case_day = df.groupby("run_no")["starttime"].first()
    horizon = case_day.values + pd.Timedelta(days=7)
    terminal = sys_wv.reindex(horizon)
    initial = sys_wv.reindex(case_day.values)
    term = terminal.to_numpy()
    n_missing = int(np.isnan(term).sum())
    if n_missing:
        term = pd.Series(term).ffill().to_numpy()  # documented fallback for the final case

    feats = pd.DataFrame(index=case_day.index)
    feats["f07_terminal_wv"] = term
    feats["f07_terminal_minus_initial"] = term - initial.to_numpy()
    feats["f07_terminal_over_initial"] = term / np.where(initial.to_numpy() == 0, np.nan,
                                                          initial.to_numpy())

    # The documented horizon is start+7, but every result up to log 032 was built on start+8,
    # and the water value series is smooth enough that the two are highly correlated. Rather
    # than assume the doc settles it empirically, emit the neighbouring lags as well and let
    # the model choose; the difference between them is itself the local slope of the water
    # value at the horizon, which is what "is water getting scarcer" means.
    for lag in (6, 8, 9):
        t_l = sys_wv.reindex(case_day.values + pd.Timedelta(days=lag)).to_numpy()
        if np.isnan(t_l).any():
            t_l = pd.Series(t_l).ffill().to_numpy()
        feats[f"f07_terminal_wv_lag{lag}"] = t_l
        feats[f"f07_terminal_wv_lag{lag}_minus_d7"] = t_l - term
    feats["f07_terminal_slope"] = feats["f07_terminal_wv_lag8"] - feats["f07_terminal_wv_lag6"]
    # per-reservoir terminal values: the marginal cost of water for the plant drawing on it
    p2r = plant_to_reservoir()
    for plant, r in p2r.items():
        col = sanitize(r)
        if r not in wv.columns:
            continue
        t_r = wv[r].reindex(horizon).to_numpy()
        i_r = wv[r].reindex(case_day.values).to_numpy()
        if np.isnan(t_r).any():
            t_r = pd.Series(t_r).ffill().to_numpy()
        feats[f"f07_terminal_wv_{col}"] = t_r
        feats[f"f07_terminal_minus_initial_{col}"] = t_r - i_r
    feats.attrs["n_terminal_missing"] = n_missing
    res_df = df[["run_no"]].merge(feats, left_on="run_no", right_index=True, how="left") \
                            .drop(columns="run_no").set_index(df.index)
    res_df["f07_price_minus_terminal_wv"] = df["price_eur_mwh"] - res_df["f07_terminal_wv"]
    # the economically correct per-plant spread: price against THIS plant's terminal water value
    for plant, r in p2r.items():
        col = sanitize(r)
        if f"f07_terminal_wv_{col}" in res_df.columns:
            res_df[f"f07_term_spread_{sanitize(plant)}"] = (
                df["price_eur_mwh"] - res_df[f"f07_terminal_wv_{col}"])
    return res_df


def stage_water_budget(df: pd.DataFrame) -> pd.DataFrame:
    """The quantity SHOP actually resolves: available water vs. profitable hours.

    SHOP maximises revenue - start/stop costs + terminal water value. The binding tension is that
    a week contains more profitable hours than the water supports, so it must ration. We give the
    model that trade-off directly:

      energy available  = (usable storage + in-horizon inflow) x MWh/m3   [head-derived]
      hours affordable  = energy available / plant capacity
      merit dispatch    = price_rank_in_case vs. hours affordable

    `merit_margin` is effectively the greedy solution to the optimizer's own problem: run the
    N most expensive hours the water can pay for. Its sign is a direct run/don't-run signal.

    Requires stage_physics to have run (uses the f11_mwh_per_m3_* conversion factors).
    """
    topo = load_topology()
    gens, reservoirs = topo["model"]["generator"], topo["model"]["reservoir"]
    p2r = plant_to_reservoir()
    out = pd.DataFrame(index=df.index)

    # in-horizon inflow: total arriving over each case's own 168h window, per reservoir
    inflow_cols = {c[len("inflow_"):]: c for c in df.columns if c.startswith("inflow_")}
    case_inflow = df.groupby("run_no")[list(inflow_cols.values())].sum()  # m3/s summed over 168h

    # price rank within the case, 1 = most expensive hour of the week
    price_rank = df.groupby("run_no")["price_eur_mwh"].rank(ascending=False, method="first")
    out["f20_price_rank_desc"] = price_rank

    for plant, r in p2r.items():
        pl = sanitize(plant)
        scol, conv_col = f"storage_{sanitize(r)}", f"f11_mwh_per_m3_{pl}"
        if scol not in df.columns or conv_col not in df.columns:
            continue
        spec = reservoirs[r]
        vh = spec["vol_head"]
        lrl_vol = float(np.interp(spec["lrl"], vh["y"], vh["x"]))  # dead storage below lrl

        usable_Mm3 = (df[scol] - lrl_vol).clip(lower=0)
        icol = inflow_cols.get(sanitize(r))
        inflow_Mm3 = (df["run_no"].map(case_inflow[icol]) * 3600 / 1e6) if icol else 0.0

        energy_mwh = (usable_Mm3 + inflow_Mm3) * 1e6 * df[conv_col]
        pgens = [g for g in gens if sanitize(g).startswith(pl + "_G")]
        capacity_mw = sum(gens[g]["p_max"] for g in pgens) or np.nan

        hours_affordable = energy_mwh / capacity_mw
        out[f"f20_energy_avail_{pl}"] = energy_mwh
        out[f"f20_hours_affordable_{pl}"] = hours_affordable
        # the greedy dispatch signal: is this hour inside the affordable top-N of the week?
        out[f"f20_merit_margin_{pl}"] = hours_affordable.clip(upper=168) - price_rank
        out[f"f20_in_merit_{pl}"] = (out[f"f20_merit_margin_{pl}"] > 0).astype(int)
        # how tight the week is: water supply relative to the whole horizon
        out[f"f20_scarcity_{pl}"] = (hours_affordable / 168.0).clip(upper=5)

    return out


def stage_wv_structure(df: pd.DataFrame) -> pd.DataFrame:
    """Theme 5: water value as one common factor plus per-reservoir deviations (PC1 = 91%)."""
    wv_cols = [c for c in df.columns if c.startswith("water_value_")]
    X = df[wv_cols]
    sys_wv = X.mean(axis=1)
    out = pd.DataFrame(index=df.index)
    out["f05_sys_wv"] = sys_wv
    out["f05_wv_dispersion"] = X.std(axis=1)
    for plant, r in plant_to_reservoir().items():
        col = f"water_value_{sanitize(r)}"
        if col in df.columns:
            out[f"f05_wv_dev_{sanitize(plant)}"] = df[col] - sys_wv
    out["f05_wv_rank_spread"] = X.rank(axis=1).std(axis=1)
    return out


## 3. Count-table features: one row per (case, unit)

Multi-horizon trailing and forward windows of price, inflow and storage; cascade water balance
and implied production per plant; constraint series; unit-relative (`own_`, `upstream_`) views.
The forward windows read the observed storage trajectory, which the volume file provides for the
test period too.

In [4]:
def bin_count(count: pd.Series) -> pd.Series:
    """Map hours-on (0..168) to one of six ordered bins."""
    c = count.to_numpy()
    b = np.full(len(c), 2, dtype=int)
    b[c == 0] = 0
    b[(c >= 1) & (c < 29)] = 1
    b[(c >= 29) & (c < 85)] = 2
    b[(c >= 85) & (c < 141)] = 3
    b[(c >= 141) & (c < 168)] = 4
    b[c == 168] = 5
    return pd.Series(b, index=count.index)


K = _ROOT / "data/kernel"


def _daily(path, how="mean") -> pd.DataFrame:
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["date"]).dt.tz_localize(None)
    d = d.set_index("date").sort_index()
    return d.resample("D").mean() if how == "mean" else d.resample("D").sum()


def base_series() -> dict[str, pd.Series]:
    """System-level daily series, plus the spread that every surviving feature is built from."""
    price = _daily(K / "Historical_day_ahead_price_2015_2025.csv").iloc[:, 0]
    wv = _daily(K / "Synthetic_water_value_2015_2024.csv")
    vol = _daily(K / "Historical_volume_2015_2024.csv")
    inf = _daily(K / "Historical_inflow_1958_2025.csv")
    res = list(vol.columns)
    sys_wv = wv[res].mean(axis=1)
    idx = price.index
    out = {
        "price": price,
        "wv": sys_wv.reindex(idx).ffill(),
        "spread": price - sys_wv.reindex(idx).ffill(),
        "storage": vol[res].sum(axis=1).reindex(idx).ffill(),
        "inflow": inf[[c for c in inf.columns if c in res]].sum(axis=1).reindex(idx).ffill(),
    }
    # inferred demand proxies: SHOP sees demand, we infer it from price behaviour
    out["price_vol"] = price.rolling(7, min_periods=2).std()
    return out


WINDOWS_BACK = {"1w": 7, "1m": 30, "3m": 90, "1y": 365}


WINDOWS_FWD = {"f1w": 7, "f1m": 30}


def horizon_frame(series: pd.Series, at_dates: pd.Series, name: str) -> pd.DataFrame:
    """Level in the case week, in windows before and after it, and every relative comparison."""
    s = series.sort_index()
    d = pd.to_datetime(pd.Series(at_dates)).reset_index(drop=True)
    out = pd.DataFrame(index=range(len(d)))

    # the case's own week
    wk = s.rolling(7, min_periods=1).mean().shift(-6)     # mean of [d, d+6]
    out[f"{name}_week"] = wk.reindex(d).to_numpy()

    # trailing windows (strictly before the case)
    for tag, n in WINDOWS_BACK.items():
        b = s.rolling(n, min_periods=2).mean().shift(1)
        out[f"{name}_back_{tag}"] = b.reindex(d).to_numpy()

    # forward windows (after the case week) - legal, and the storability argument makes them matter
    for tag, n in WINDOWS_FWD.items():
        f = s.rolling(n, min_periods=2).mean().shift(-(n + 6))
        out[f"{name}_{tag}"] = f.reindex(d).to_numpy()

    # previous and next week explicitly
    out[f"{name}_prev_week"] = s.rolling(7, min_periods=1).mean().shift(1).reindex(d).to_numpy()
    out[f"{name}_next_week"] = s.rolling(7, min_periods=1).mean().shift(-13).reindex(d).to_numpy()

    # relative comparisons - what actually carries signal across regimes
    w = out[f"{name}_week"]
    for tag in list(WINDOWS_BACK) + list(WINDOWS_FWD):
        col = f"{name}_back_{tag}" if tag in WINDOWS_BACK else f"{name}_{tag}"
        out[f"{name}_vs_{tag}"] = w - out[col]
        denom = out[col].replace(0, np.nan)
        out[f"{name}_ratio_{tag}"] = w / denom
    out[f"{name}_vs_prev_week"] = w - out[f"{name}_prev_week"]
    out[f"{name}_vs_next_week"] = w - out[f"{name}_next_week"]
    out[f"{name}_next_minus_prev"] = out[f"{name}_next_week"] - out[f"{name}_prev_week"]

    # seasonal position: this week against the same calendar week across all history
    doy = pd.Series(s.index.dayofyear, index=s.index)
    seas = s.groupby(doy).transform("mean")
    out[f"{name}_vs_seasonal"] = w - seas.reindex(d).to_numpy()
    return out


def build_count_features(at_dates: pd.Series) -> pd.DataFrame:
    """All horizons for all system series, indexed to match `at_dates` (one row per case)."""
    ser = base_series()
    frames = [horizon_frame(s, at_dates, n) for n, s in ser.items()]
    return pd.concat(frames, axis=1)


def _through_tunnels(c, node, reservoirs, depth=6):
    """Reservoirs feeding `node`, walking back through any chain of tunnel hops.

    The topology connects some reservoirs to plants DIRECTLY and others through a named tunnel:

        Vinjevatn -------------------------------> Tokke     (direct)
        Songavatn --> Songavatn_Songa -----------> Songa      (via tunnel)

    Matching only direct reservoir->plant edges therefore misses every tunnel-fed plant. That is
    not hypothetical: it left `src["Songa"] = None`, so every `own_*` feature for Songa was NaN
    and `mwh_per_m3("Songa", .)` returned NaN -- while Songavatn carries the largest storage
    variance of any reservoir in the system (sd 150) and Bitdalsvatn adds another (sd 25). Log
    027 diagnosed Songa as "tunnel-fed" and attacked it with water-balance corrections; the
    actual defect was here, in the graph.
    """
    out, seen, frontier = set(), set(), {node}
    for _ in range(depth):
        nxt = set()
        for x in c:
            if x["to"] in frontier and x["from"] not in seen:
                seen.add(x["from"])
                if x["from"] in reservoirs:
                    out.add(x["from"])
                elif x["from_type"] == "tunnel":
                    nxt.add(x["from"])
        frontier = nxt
        if not frontier:
            break
    return out


_topo = None


def topo():
    global _topo
    if _topo is None:
        _topo = yaml.safe_load(open(_ROOT / "data/extended/Tokke_Vinje_topology.yaml"))
    return _topo


def plant_sources():
    """Every source reservoir of every plant, tunnels resolved. Songa has two."""
    t = topo(); c = t["connections"]; res = set(t["model"]["reservoir"])
    return {p: sorted(_through_tunnels(c, p, res)) for p in t["model"]["plant"]}


def plant_graph():
    """source reservoir, discharge reservoir, and upstream set for each plant.

    `src` keeps its single-reservoir contract for existing callers, but is now resolved through
    tunnels; where a plant has several sources the largest-capacity one is primary and
    `plant_sources()` exposes the rest.
    """
    t = topo(); c = t["connections"]; res = set(t["model"]["reservoir"])
    direct = {x["to"]: x["from"] for x in c
              if x["to_type"] == "plant" and x["from_type"] == "reservoir"}
    allsrc = plant_sources()

    def cap(r):
        vh = t["model"]["reservoir"][r].get("vol_head", {})
        return max(vh.get("x", [0.0])) if vh else 0.0

    src = dict(direct)
    for p in t["model"]["plant"]:
        if p not in src and allsrc.get(p):
            src[p] = max(allsrc[p], key=cap)
    dst = {x["from"]: x["to"] for x in c if x["from_type"] == "plant" and x["to_type"] == "reservoir"}
    up = {}
    for p in t["model"]["plant"]:
        seen, frontier = set(), {p}
        for _ in range(4):
            nxt = set()
            for x in c:
                if x["to"] in frontier and x["from"] not in seen:
                    seen.add(x["from"]); nxt.add(x["from"])
            frontier = nxt
        up[p] = {r for r in seen if r in t["model"]["reservoir"]}
    return src, dst, up


def san(s):
    return s.replace(" ", "_")


def mwh_per_m3(plant, storage_value):
    """Head-derived conversion: storage -> level -> head above outlet -> energy per m3."""
    t = topo()
    src, _, _ = plant_graph()
    r = src.get(plant)
    if r is None:
        return np.nan
    spec = t["model"]["reservoir"][r]; vh = spec["vol_head"]
    level = np.interp(storage_value, vh["x"], vh["y"])
    head = level - t["model"]["plant"][plant]["outlet_line"]
    gens = [g for g in t["model"]["generator"] if san(g).startswith(san(plant) + "_G")]
    eff = 85.0
    if gens:
        curves = t["model"]["generator"][gens[0]].get("turb_eff_curves", [])
        if curves:
            refs = np.array([c["ref"] for c in curves], float)
            peaks = np.array([max(c["y"]) for c in curves], float)
            o = np.argsort(refs); eff = np.interp(head, refs[o], peaks[o])
    return 1000 * 9.81 * head * (eff / 100.0) / 3.6e9


def pf_build(at_dates: pd.Series) -> pd.DataFrame:
    """One row per case; columns are per-plant water-balance and binding-constraint features."""
    price = _daily(K / "Historical_day_ahead_price_2015_2025.csv").iloc[:, 0]
    wv = _daily(K / "Synthetic_water_value_2015_2024.csv")
    vol = _daily(K / "Historical_volume_2015_2024.csv")
    inf_d = _daily(K / "Historical_inflow_1958_2025.csv") * 3600 * 24 / 1e6   # m3/s -> Mm3/day
    src, dst, up = plant_graph()
    t = topo(); gens = t["model"]["generator"]; res_spec = t["model"]["reservoir"]

    d = pd.to_datetime(pd.Series(at_dates)).reset_index(drop=True)
    d_end = d + pd.Timedelta(days=7)
    out = pd.DataFrame(index=range(len(d)))

    def at(s, when):
        return s.reindex(when).to_numpy()

    def win_sum(s, a, b):
        c = s.cumsum()
        return c.reindex(b).to_numpy() - c.reindex(a).to_numpy()

    # ---------- system-level balance (the crude proxy that already wins) ----------
    sys_stor = vol.sum(axis=1)
    sys_inf = inf_d[[c for c in inf_d.columns if c in vol.columns]].sum(axis=1)
    out["sys_drawdown"] = at(sys_stor, d) - at(sys_stor, d_end)
    out["sys_inflow_wk"] = win_sum(sys_inf, d, d_end)
    out["sys_release"] = out["sys_drawdown"] + out["sys_inflow_wk"]
    out["sys_release_over_inflow"] = out["sys_release"] / out["sys_inflow_wk"].replace(0, np.nan)
    out["sys_drawdown_frac"] = out["sys_drawdown"] / at(sys_stor, d)

    for plant, r in src.items():
        pl = san(plant)
        if r not in vol.columns:
            continue
        stor = vol[r]; inflow = inf_d[r] if r in inf_d.columns else pd.Series(0.0, index=vol.index)
        s0, s1 = at(stor, d), at(stor, d_end)
        inflow_wk = win_sum(inflow, d, d_end)

        # upstream plants discharge INTO this reservoir -- the term the naive attempt omitted
        up_disch = np.zeros(len(d))
        for p2, dr in dst.items():
            if dr == r and p2 != plant and src.get(p2) in vol.columns:
                s2 = vol[src[p2]]
                i2 = inf_d[src[p2]] if src[p2] in inf_d.columns else pd.Series(0.0, index=vol.index)
                up_disch += (at(s2, d) - at(s2, d_end)) + win_sum(i2, d, d_end)

        release = (s0 - s1) + inflow_wk + up_disch
        conv = mwh_per_m3(plant, s0)
        cap = sum(gens[g]["p_max"] for g in gens if san(g).startswith(pl + "_G")) or np.nan
        n_units = sum(1 for g in gens if san(g).startswith(pl + "_G"))
        energy = release * 1e6 * conv                       # MWh released this week

        out[f"drawdown_{pl}"] = s0 - s1
        out[f"release_{pl}"] = release
        out[f"upstream_discharge_{pl}"] = up_disch
        out[f"energy_mwh_{pl}"] = energy
        # implied hours at full output -- a DIRECT physical estimate of the count
        out[f"implied_hours_{pl}"] = energy / cap
        out[f"capacity_factor_{pl}"] = energy / (cap * 168.0)
        out[f"implied_unit_hours_{pl}"] = energy / (cap / max(n_units, 1))

        # normalisations
        mv = res_spec[r]["max_vol"]
        out[f"stor_frac_{pl}"] = s0 / mv
        out[f"stor_frac_end_{pl}"] = s1 / mv
        out[f"release_over_maxvol_{pl}"] = release / mv
        out[f"release_over_inflow_{pl}"] = release / np.where(inflow_wk == 0, np.nan, inflow_wk)
        out[f"inflow_over_maxvol_{pl}"] = inflow_wk / mv

        # ---------- the two limits, and which one binds ----------
        own_wv = wv[r].reindex(price.index).ffill() if r in wv.columns else wv.mean(axis=1)
        spread = price - own_wv
        prof_hours = (spread > 0).astype(float)
        hours_profitable = win_sum(prof_hours, d, d_end) * 24     # daily series -> hours
        water_hours = out[f"implied_hours_{pl}"]
        out[f"hours_profitable_{pl}"] = hours_profitable
        out[f"hours_water_{pl}"] = water_hours
        out[f"binding_min_{pl}"] = np.minimum(hours_profitable, water_hours)
        out[f"water_over_profitable_{pl}"] = water_hours / np.where(hours_profitable == 0, np.nan,
                                                                     hours_profitable)
        out[f"water_is_binding_{pl}"] = (water_hours < hours_profitable).astype(float)
        out[f"mean_spread_{pl}"] = win_sum(spread.fillna(0), d, d_end) / 7.0
    return out


def water_value_trace(at_dates: pd.Series) -> pd.DataFrame:
    wv = _daily(K / "Synthetic_water_value_2015_2024.csv")
    price = _daily(K / "Historical_day_ahead_price_2015_2025.csv").iloc[:, 0]
    src, dst, up = plant_graph()
    d = pd.to_datetime(pd.Series(at_dates)).reset_index(drop=True)
    d1 = d + pd.Timedelta(days=7)
    d2 = d + pd.Timedelta(days=14)
    dm1 = d - pd.Timedelta(days=7)
    out = {}

    sysw = wv.mean(axis=1)
    out["wvt_sys_change"] = sysw.reindex(d1).to_numpy() - sysw.reindex(d).to_numpy()
    out["wvt_sys_change_prev"] = sysw.reindex(d).to_numpy() - sysw.reindex(dm1).to_numpy()
    out["wvt_sys_change_next"] = sysw.reindex(d2).to_numpy() - sysw.reindex(d1).to_numpy()
    out["wvt_sys_accel"] = out["wvt_sys_change"] - out["wvt_sys_change_prev"]
    out["wvt_sys_dispersion_change"] = (wv.std(axis=1).reindex(d1).to_numpy()
                                        - wv.std(axis=1).reindex(d).to_numpy())

    for plant, r in src.items():
        if r not in wv.columns:
            continue
        pl = san(plant)
        w0, w1 = wv[r].reindex(d).to_numpy(), wv[r].reindex(d1).to_numpy()
        out[f"wvt_change_{pl}"] = w1 - w0
        out[f"wvt_change_prev_{pl}"] = w0 - wv[r].reindex(dm1).to_numpy()
        out[f"wvt_change_next_{pl}"] = wv[r].reindex(d2).to_numpy() - w1
        # relative to the system move: did THIS reservoir deplete faster than the system?
        out[f"wvt_change_vs_sys_{pl}"] = (w1 - w0) - out["wvt_sys_change"]
        # scale-free
        out[f"wvt_change_pct_{pl}"] = (w1 - w0) / np.where(np.abs(w0) < 1e-6, np.nan, np.abs(w0))
        # against the price it was competing with
        p_wk = price.rolling(7, min_periods=1).mean().shift(-6).reindex(d).to_numpy()
        out[f"wvt_change_over_price_{pl}"] = (w1 - w0) / np.where(p_wk == 0, np.nan, p_wk)
        # downstream reservoir's water value also moves when this plant discharges into it
        dr = dst.get(plant)
        if dr in wv.columns:
            out[f"wvt_downstream_change_{pl}"] = (wv[dr].reindex(d1).to_numpy()
                                                   - wv[dr].reindex(d).to_numpy())
    return pd.DataFrame(out)


def add_constraint_series(dates: pd.Series) -> pd.DataFrame:
    """Weekly mean of each constraint series, one row per case."""
    out = pd.DataFrame(index=range(len(dates)))
    d = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    for tag, path in [("mv", "data/extended/Constraint_min_volume.csv"),
                      ("mf", "data/extended/Constraint_min_flow.csv")]:
        c = pd.read_csv(_ROOT / path)
        c["date"] = pd.to_datetime(c["date"]).dt.tz_localize(None)
        c = c.set_index("date").sort_index().resample("D").mean()
        wk = c.rolling(7, min_periods=1).mean().shift(-6)
        for col in c.columns:
            out[f"{tag}_{san(col)}"] = wk[col].reindex(d).to_numpy()
    return out


def relevance_map() -> dict[str, dict]:
    """Per plant: source, discharge target, upstream plants, and wired constraints."""
    t = topo(); c = t["connections"]
    src, dst, up = plant_graph()
    mv = [x for x in pd.read_csv(_ROOT / "data/extended/Constraint_min_volume.csv", nrows=1).columns
          if x != "date"]
    mf = [x for x in pd.read_csv(_ROOT / "data/extended/Constraint_min_flow.csv", nrows=1).columns
          if x != "date"]

    def down(node, depth=3):
        seen, fr = set(), {node}
        for _ in range(depth):
            nx = set()
            for x in c:
                if x["from"] in fr and x["to"] not in seen:
                    seen.add(x["to"]); nx.add(x["to"])
            fr = nx
        return seen

    out = {}
    for p in t["model"]["plant"]:
        near = up[p] | down(p)
        # plants whose discharge lands in this plant's source reservoir
        upstream_plants = [q for q, d in dst.items() if d == src.get(p) and q != p]
        out[p] = {
            "source": src.get(p),
            "discharges_to": dst.get(p),
            "upstream_plants": upstream_plants,
            "min_vol": [r for r in mv if r in near],
            "min_flow": [r for r in mf if r in near],
        }
    return out


def build_unit_relative(CU: pd.DataFrame, PROD: pd.DataFrame, CONS: pd.DataFrame,
                        units: list[str]) -> pd.DataFrame:
    """Reassemble per-plant columns into unit-relative ones. CU must carry run_no and unit."""
    rel = relevance_map()
    plant_of = {u: u.rsplit("_G", 1)[0] for u in units}
    parts = []
    for u in units:
        pl_raw = next((p for p in rel if san(p) == plant_of[u]), plant_of[u])
        info = rel.get(pl_raw, {})
        pl = san(pl_raw)
        sub = CU[CU.unit == u].copy()
        prod = PROD.reindex(sub["run_no"]).reset_index(drop=True)
        cons = CONS.reindex(sub["run_no"]).reset_index(drop=True)
        sub = sub.reset_index(drop=True)

        # own plant, generic names
        own = prod[[c for c in prod.columns if c.endswith("_" + pl)]].copy()
        own.columns = ["own_" + c[: -(len(pl) + 1)] for c in own.columns]

        # upstream plants -> their production is this plant's inflow
        ups = [san(q) for q in info.get("upstream_plants", [])]
        up_cols = {}
        for base in ("release", "energy_mwh", "implied_hours", "capacity_factor", "drawdown"):
            cs = [f"{base}_{q}" for q in ups if f"{base}_{q}" in prod.columns]
            # no upstream plant means upstream discharge is genuinely ZERO, not missing
            up_cols[f"upstream_{base}"] = (prod[cs].sum(axis=1) if cs
                                           else pd.Series(0.0, index=prod.index))
        upf = pd.DataFrame(up_cols)

        # system context, shared legitimately
        sysf = prod[[c for c in prod.columns if c.startswith("sys_")]].copy()

        # constraints actually wired to this plant
        cc = {}
        for r in info.get("min_vol", []):
            if f"mv_{san(r)}" in cons.columns:
                cc[f"ownmv_{san(r)}"] = cons[f"mv_{san(r)}"]
        for r in info.get("min_flow", []):
            if f"mf_{san(r)}" in cons.columns:
                cc[f"ownmf_{san(r)}"] = cons[f"mf_{san(r)}"]
        consf = pd.DataFrame(cc) if cc else pd.DataFrame(index=sub.index)
        consf = consf.reindex(sub.index)
        # summarise so the column set is identical across units
        consf = pd.DataFrame({
            "own_minvol_level": consf[[c for c in consf.columns if c.startswith("ownmv_")]].mean(axis=1)
            if any(c.startswith("ownmv_") for c in consf.columns) else np.nan,
            "own_minflow_level": consf[[c for c in consf.columns if c.startswith("ownmf_")]].mean(axis=1)
            if any(c.startswith("ownmf_") for c in consf.columns) else np.nan,
            "own_n_minvol": len(info.get("min_vol", [])),
            "own_n_minflow": len(info.get("min_flow", [])),
            "own_n_upstream_plants": len(ups),
        }, index=sub.index)

        parts.append(pd.concat([sub, own, upf, sysf, consf], axis=1))
    return pd.concat(parts, ignore_index=True)


## 4. Hourly block structure and the long (case, hour, unit) format

In [5]:
WINDOWS = (3, 6, 12, 24, 48, 96)


def _centered_window_mean(a: np.ndarray, w: int) -> np.ndarray:
    """Mean over a window of length w centred on each hour, edges shrinking rather than padding.

    Edge hours matter here -- hour 0 and hour 167 of a case are real decisions, not padding --
    so the window truncates at the boundary instead of wrapping or filling with zeros, which
    would invent price structure that is not there.
    """
    cs = np.r_[0.0, np.cumsum(a)]
    n = len(a)
    idx = np.arange(n)
    lo = np.maximum(idx - w // 2, 0)
    hi = np.minimum(idx + (w - w // 2), n)
    return (cs[hi] - cs[lo]) / (hi - lo)


def _feature_names() -> list[str]:
    n = ["hs_rank", "hs_z", "hs_dist_to_peak", "hs_dist_to_trough",
         "hs_run_pos", "hs_run_len", "hs_above_med", "hs_cum_share",
         "hs_hod_sin", "hs_hod_cos", "hs_dow_sin", "hs_dow_cos"]
    for w in WINDOWS:
        n += [f"hs_win{w}", f"hs_win{w}_rank", f"hs_in_best{w}"]
    for lag in (1, 3, 24):
        n += [f"hs_lag{lag}", f"hs_lead{lag}"]
    return n


def hs_build(df: pd.DataFrame, signal: str = "f06_spread_sys",
          run_col: str = "run_no", hour_col: str = "hour") -> pd.DataFrame:
    """Per-hour block-structure features, one row per (case, hour) aligned to `df`.

    All features are computed WITHIN a case and are either scale-free (ranks, z-scores) or
    differences against the case's own level, so none of them carry the absolute price level
    that log 029 showed drifts across the 2021-22 crisis.
    """
    d = df[[run_col, hour_col, signal]].copy()
    d = d.sort_values([run_col, hour_col])
    out = {k: np.empty(len(d)) for k in _feature_names()}
    pos = 0
    for _, g in d.groupby(run_col, sort=False):
        s = g[signal].to_numpy(dtype=float)
        n = len(s)
        sl = slice(pos, pos + n)
        med = np.nanmedian(s)
        sd = np.nanstd(s)
        order = np.argsort(np.argsort(-s))          # 0 = highest spread hour in the case

        out["hs_rank"][sl] = order / max(n - 1, 1)
        out["hs_z"][sl] = (s - np.nanmean(s)) / sd if sd > 0 else 0.0
        out["hs_dist_to_peak"][sl] = np.abs(np.arange(n) - int(np.argmax(s))) / n
        out["hs_dist_to_trough"][sl] = np.abs(np.arange(n) - int(np.argmin(s))) / n

        for w in WINDOWS:
            m = _centered_window_mean(s, w)
            out[f"hs_win{w}"][sl] = m - med
            r = np.argsort(np.argsort(-m)) / max(n - 1, 1)
            out[f"hs_win{w}_rank"][sl] = r
            # is this hour inside the best length-w stretch of the week?
            if n >= w:
                cs = np.r_[0.0, np.cumsum(s)]
                tot = cs[w:] - cs[:-w]
                j = int(np.argmax(tot))
                inb = np.zeros(n); inb[j:j + w] = 1.0
            else:
                inb = np.ones(n)
            out[f"hs_in_best{w}"][sl] = inb

        # run structure: how deep into a contiguous above-median stretch this hour sits
        above = (s > med).astype(int)
        run_id = np.cumsum(np.r_[1, np.diff(above) != 0])
        depth = np.zeros(n); length = np.zeros(n)
        for rid in np.unique(run_id):
            m_ = run_id == rid
            k = int(m_.sum())
            depth[m_] = np.arange(k) / max(k - 1, 1)
            length[m_] = k / n
        out["hs_run_pos"][sl] = depth
        out["hs_run_len"][sl] = length
        out["hs_above_med"][sl] = above

        # lags and leads within the case, as differences so no level leaks in
        for lag in (1, 3, 24):
            out[f"hs_lag{lag}"][sl] = s - np.r_[np.full(min(lag, n), s[0]), s[:-lag]][:n]
            out[f"hs_lead{lag}"][sl] = s - np.r_[s[lag:], np.full(min(lag, n), s[-1])][:n]

        # cumulative share of the week's positive spread consumed before this hour
        pos_s = np.clip(s, 0, None)
        tot = pos_s.sum()
        out["hs_cum_share"][sl] = (np.cumsum(pos_s) / tot) if tot > 0 else np.linspace(0, 1, n)

        # calendar position, cyclically encoded so hour 23 and hour 0 are adjacent
        hod = g[hour_col].to_numpy() % 24
        out["hs_hod_sin"][sl] = np.sin(2 * np.pi * hod / 24)
        out["hs_hod_cos"][sl] = np.cos(2 * np.pi * hod / 24)
        dow = (g[hour_col].to_numpy() // 24) % 7
        out["hs_dow_sin"][sl] = np.sin(2 * np.pi * dow / 7)
        out["hs_dow_cos"][sl] = np.cos(2 * np.pi * dow / 7)
        pos += n

    res = pd.DataFrame(out, index=d.index).reindex(df.index)
    return res


_OWN = ("p_off", "p_on", "p_low", "p_high", "exp_count", "margin",
        # full-distribution outputs (count_model.distribution_outputs); p_off/p_on above are
        # the exact all-off and all-on endpoint bins, so only the interior bins are listed here
        "p_bin1", "p_bin2", "p_bin3", "p_bin4",
        "count_sd", "entropy", "mode_bin", "p_lt_quarter", "p_ge_half")


def to_long(wide: pd.DataFrame, units: list[str], shared: list[str],
            count_prefix: str = "cnt_", coupled_prefix: tuple[str, ...] = ("hs_", "nb_"),
            static: pd.DataFrame | None = None,
            case_stride: int = 1, run_col: str = "run_no",
            dtype=np.float32) -> pd.DataFrame:
    """Melt a wide (case, hour) frame into (case, hour, unit) rows.

    `shared` columns are copied to every unit's rows unchanged -- they are genuine case/hour
    context (system spread, hour-of-day, window aggregates). Per-unit columns named
    `<prefix><field>_<unit>` collapse into generic `own_<field>` columns, which is the whole
    point: the model sees "my own predicted count", not "Lio's predicted count".
    """
    w = wide
    if case_stride > 1:
        keep = np.sort(w[run_col].unique())[::case_stride]
        w = w[w[run_col].isin(keep)]

    shared = [c for c in shared if c in w.columns]
    base = w[[run_col, "hour"] + shared].reset_index(drop=True)
    base[shared] = base[shared].astype(dtype)

    parts = []
    for u in units:
        own = {}
        for f in _OWN:
            c = f"{count_prefix}{f}_{u}"
            if c in w.columns:
                own[f"own_{f}"] = w[c].to_numpy(dtype=dtype)
        prefixes = (coupled_prefix,) if isinstance(coupled_prefix, str) else coupled_prefix
        for c in w.columns:
            for pre in prefixes:
                if c.startswith(pre) and c.endswith("_" + u):
                    own["own_" + c[len(pre):-(len(u) + 1)]] = w[c].to_numpy(dtype=dtype)
                    break
        p = pd.concat([base, pd.DataFrame(own)], axis=1)
        p["unit"] = u
        p["y"] = w[f"on_{u}"].to_numpy(dtype=np.int8)
        parts.append(p)

    out = pd.concat(parts, ignore_index=True)
    if static is not None:
        out = out.merge(static, on="unit", how="left")
    out["unit"] = out["unit"].astype("category")
    return out


## 5. Pipeline: features → count-model ranking → long format → XGBoost → submission

The count model is fitted only to rank the count-table features; its predictions are not
forwarded (forwarding them cost 0.008, log 035).

In [6]:
def long_matrix(w, units, shared, static, ctx, keep, case_stride=1):
    '''(case, hour, unit) design matrix, as `to_long` + the cf_ block, without pandas copies.

    Columns: shared, own_*, unit (categorical code), static attributes, cf_* count features --
    the order `to_long(...)` then `concat([L, blk])` produced. Rows: unit-major, cases in order.
    '''
    if case_stride > 1:
        sel = np.sort(w["run_no"].unique())[::case_stride]
        w = w[w["run_no"].isin(sel)]
    shared = [c for c in shared if c in w.columns]
    own_src, own_names = {}, []
    for u in units:
        src = {}
        for f in _OWN:
            if f"cnt_{f}_{u}" in w.columns:
                src[f"own_{f}"] = f"cnt_{f}_{u}"
        for c in w.columns:
            for pre in ("hs_", "nb_"):
                if c.startswith(pre) and c.endswith("_" + u):
                    src["own_" + c[len(pre):-(len(u) + 1)]] = c
                    break
        own_src[u] = src
        own_names += [k for k in src if k not in own_names]
    stat_cols = [c for c in static.columns if c != "unit"]
    cols = shared + own_names + ["unit"] + stat_cols + [f"cf_{c}" for c in keep]
    ftypes = ["c" if c == "unit" else "q" for c in cols]
    cats = sorted(units)                          # pandas orders string categories this way
    n, U = len(w), len(units)
    X = np.full((n * U, len(cols)), np.nan, np.float32)
    y = np.empty(n * U, np.float32)
    run = np.tile(w["run_no"].to_numpy(), U)
    hour = np.tile(w["hour"].to_numpy(), U)
    ucode = np.repeat([cats.index(u) for u in units], n).astype(np.int16)
    ns, no, ju = len(shared), len(own_names), len(shared) + len(own_names)
    shared_block = w[shared].to_numpy(dtype=np.float32)
    st = static.set_index("unit")[stat_cols]
    for j, u in enumerate(units):
        r = slice(j * n, (j + 1) * n)
        X[r, :ns] = shared_block
        for k, c in own_src[u].items():
            X[r, ns + own_names.index(k)] = w[c].to_numpy(dtype=np.float32)
        X[r, ju] = cats.index(u)
        X[r, ju + 1:ju + 1 + len(stat_cols)] = st.loc[u].to_numpy(dtype=np.float32)
        idx = pd.MultiIndex.from_arrays([w["run_no"].to_numpy(), np.full(n, u, dtype=object)])
        X[r, ju + 1 + len(stat_cols):] = ctx.reindex(idx).to_numpy(dtype=np.float32)
        y[r] = w[f"on_{u}"].to_numpy(dtype=np.float32) if f"on_{u}" in w.columns else np.nan
    return run, hour, ucode, X, y, cols, ftypes, cats


COUNT_PARAMS = dict(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.6, tree_method="hist", n_jobs=-1, random_state=0)


N_CORE, N_UNITREL = 120, 30


POINT_PARAMS = dict(max_depth=8, eta=0.05, subsample=0.8, colsample_bytree=0.8,
                    min_child_weight=500, tree_method="hist", max_bin=256,
                    objective="binary:logistic", eval_metric="logloss", nthread=-1, seed=0)


POINT_ROUNDS = 600


def build_count_table(df: pd.DataFrame, units: list[str]) -> pd.DataFrame:
    """The (case, unit) table the count model ranks features on."""
    dates = df.groupby("run_no")["starttime"].first()
    sys_ = build_count_features(dates); sys_.index = dates.index
    prod = pf_build(dates); prod.index = dates.index
    cons = add_constraint_series(dates); cons.index = dates.index
    wvt = water_value_trace(dates); wvt.index = dates.index

    rows = []
    g = df.groupby("run_no")
    for u in units:
        pl = u.rsplit("_G", 1)[0]
        col = f"f06_spread_{pl}" if f"f06_spread_{pl}" in df.columns else "f06_spread_sys"
        gs = g[col]
        r = pd.DataFrame({"run_no": g["year"].first().index, "unit": u,
                          "spread_mean": gs.mean().values, "spread_std": gs.std().values,
                          "spread_max": gs.max().values, "spread_min": gs.min().values})
        cnt = g[f"on_{u}"].sum()
        r["count"] = cnt.values
        rows.append(r)
    cu = pd.concat(rows, ignore_index=True)
    return (build_unit_relative(cu.merge(sys_, left_on="run_no", right_index=True, how="left"),
                                   prod, cons, units)
            .merge(prod.add_prefix("abs_"), left_on="run_no", right_index=True, how="left")
            .merge(cons.add_prefix("c_"), left_on="run_no", right_index=True, how="left")
            .merge(wvt, left_on="run_no", right_index=True, how="left")
            .replace([np.inf, -np.inf], np.nan))


STAGES: list[tuple[str, callable]] = [
    ("01_seasonal",         stage_seasonal),
    ("01g_seasonal_global", stage_seasonal_global),
    ("09_inflow_regime",    stage_inflow_regime),
    ("02_price_relative",   stage_price_relative),
    ("06_spread",           stage_spread),
    ("07_terminal_wv",      stage_terminal_wv),
    ("10_12_physics",       stage_physics),
    ("20_water_budget",     stage_water_budget),
    ("06x_spread_extended", stage_spread_extended),
    ("04_price_shape",      stage_price_shape),
    ("03_price_windows",    stage_price_windows),
    ("05_wv_structure",     stage_wv_structure),
    ("15_merit_order",      stage_merit_order),
    ("14_graph_cascade",    stage_graph_cascade),
    ("17_calendar",         stage_calendar),
]


def build_features(verbose=True) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Feature stages over train and test together, then split back.

    Built jointly because several stages compare a case against the whole record (seasonal
    percentiles, trailing windows); computing them on train alone and test alone would put the
    two on different scales.
    """
    tr, te = build_base_tables()
    targets = [c for c in tr.columns if c.startswith("on_")]
    units = [t[3:] for t in targets]
    te = te.assign(**{c: np.nan for c in targets})
    both = pd.concat([tr, te], ignore_index=True)
    both["year"] = both["starttime"].dt.year
    both["_is_test"] = np.r_[np.zeros(len(tr), bool), np.ones(len(te), bool)]
    for name, fn in STAGES:
        t0 = time.time()
        out = fn(both)
        new = [c for c in out.columns if c not in both.columns]
        both = pd.concat([both, out[new]], axis=1)
        if verbose:
            print(f"  {name:<22} +{len(new):>3} cols  ({time.time()-t0:.1f}s)", flush=True)
    m = both["_is_test"].to_numpy()
    return both[~m].reset_index(drop=True), both[m].reset_index(drop=True), units


def submission_columns(units: list[str]) -> dict[str, str]:
    """Map sanitised unit -> the generator name the submission file uses.

    `sanitize` turns "Vesle Kjela_G1" into "Vesle_Kjela_G1", and the submission wants the
    original. Inverting by replacing underscores would also corrupt the `_G1` suffix, so the
    mapping is taken from the topology rather than reconstructed.
    """
    gens = load_topology()["model"]["generator"]
    return {sanitize(g): g for g in gens if sanitize(g) in set(units)}


def run(stride: int = 1, out_path: str | None = None, verbose: bool = True) -> pd.DataFrame:
    """Fit on all training cases, predict the 725 test cases, write the wide submission."""
    from xgboost import QuantileDMatrix, XGBClassifier, train as xgb_train

    t0 = time.time()
    tr, te, units = build_features(verbose=verbose)
    targets = [f"on_{u}" for u in units]
    print(f"features built: train {tr.shape}, test {te.shape} | {time.time()-t0:.0f}s", flush=True)

    both = pd.concat([tr, te], ignore_index=True)
    ct = build_count_table(both, units)
    del both
    gc.collect()
    ct["bin"] = bin_count(ct["count"].fillna(0))
    feats = [c for c in ct.columns if c not in ("run_no", "unit", "count", "cls", "bin")]
    train_runs = set(tr["run_no"].unique())
    ct_tr = ct[ct.run_no.isin(train_runs) & ct["count"].notna()]
    print(f"count table {ct.shape} | ranking on {len(ct_tr):,} train unit-weeks "
          f"| {time.time()-t0:.0f}s", flush=True)

    # the count model is fitted ONLY to rank features; its predictions are never forwarded
    seen = sorted(ct_tr["bin"].unique())
    cm = XGBClassifier(objective="multi:softprob", num_class=len(seen),
                       eval_metric="mlogloss", **COUNT_PARAMS)
    cm.fit(ct_tr[feats], ct_tr["bin"].map({b: i for i, b in enumerate(seen)}))
    imp = pd.Series(cm.feature_importances_, index=feats).sort_values(ascending=False)
    keep = list(imp.head(N_CORE).index)
    keep += [c for c in imp.index[N_CORE:] if c.startswith(("own_", "upstream_"))][:N_UNITREL]
    print(f"selected {len(keep)} count features | {time.time()-t0:.0f}s", flush=True)

    hs_tr = hs_build(tr); tr = pd.concat([tr, hs_tr], axis=1)
    hs_te = hs_build(te); te = pd.concat([te, hs_te], axis=1)
    shared = [c for c in hs_tr.columns]
    pruned = list(PRUNED_FEATURES)
    tw = [c for c in tr.columns if c.startswith("f07_term_spread_")
          or c in ("f07_terminal_slope", "f07_terminal_wv_lag8_minus_d7")]
    shared = [c for c in pruned + tw + shared if c in tr.columns and c in te.columns]

    gens = load_topology()["model"]["generator"]
    merit = {"Tokke_G1": 1, "Tokke_G3": 2, "Tokke_G4": 3, "Tokke_G2": 4,
             "Vinje_G1": 1, "Vinje_G2": 2, "Vinje_G3": 3}
    stat = pd.DataFrame([{
        "unit": u,
        "s_pmax": gens[[k for k in gens if sanitize(k) == u][0]]["p_max"],
        "s_pmin": gens[[k for k in gens if sanitize(k) == u][0]]["p_min"],
        "s_merit": merit.get(u, 1),
        "s_nsib": sum(1 for v in units if v.rsplit("_G", 1)[0] == u.rsplit("_G", 1)[0]),
    } for u in units])

    for d in (tr, te):
        d.replace([np.inf, -np.inf], np.nan, inplace=True)
    ctx = ct.set_index(["run_no", "unit"])[keep]

    L_run, L_hour, L_unit, Xtr, ytr, cols, ftypes, cats = long_matrix(
        tr, units, shared, stat, ctx, keep, stride)
    print(f"long: train {Xtr.shape[0]:,} rows, {len(cols)} features | {time.time()-t0:.0f}s",
          flush=True)
    del tr, hs_tr                                 # not used again; frees ~1.7 GB before the peak
    gc.collect()
    dtr = QuantileDMatrix(Xtr, label=ytr, feature_names=cols, feature_types=ftypes,
                          enable_categorical=True, max_bin=POINT_PARAMS["max_bin"])
    del Xtr, L_run, L_hour, L_unit
    gc.collect()
    bst = xgb_train(POINT_PARAMS, dtr, num_boost_round=POINT_ROUNDS)
    print(f"fitted {POINT_ROUNDS} rounds | {time.time()-t0:.0f}s", flush=True)
    V_run, V_hour, V_unit, Xte, _, _, _, _ = long_matrix(te, units, shared, stat, ctx, keep, 1)
    dva = QuantileDMatrix(Xte, feature_names=cols, feature_types=ftypes, enable_categorical=True,
                          ref=dtr, max_bin=POINT_PARAMS["max_bin"])
    V = pd.DataFrame({"run_no": V_run, "hour": V_hour,
                      "unit": pd.Categorical.from_codes(V_unit, categories=cats)})
    V["p"] = bst.predict(dva)

    # wide: one row per case, `result_committed_<Plant>_<Unit>_t<hour>`
    name = submission_columns(units)
    wide = V.pivot_table(index="run_no", columns=["unit", "hour"], values="p", observed=True)
    # Emit columns in the SAMPLE's order, not ours. The sample orders units as the topology
    # does (Hogga first, Vesle Kjela last); `units` comes from the parquet, which is
    # alphabetical. Building from the sample's header makes the order correct by construction
    # instead of by coincidence.
    sample = pd.read_csv(_ROOT / "sample_submission.csv")
    inv = {v: k for k, v in name.items()}
    cols = {}
    for c in sample.columns:
        if c == "Run No":
            continue
        m = re.fullmatch(r"result_committed_(.+)_t(\d+)", c)
        gen, h = m.group(1), int(m.group(2))
        cols[c] = wide[(inv[gen], h)].to_numpy()
    out = pd.DataFrame(cols, index=wide.index)
    out.index.name = "Run No"
    out = out.reset_index()

    assert list(out.columns) == list(sample.columns), "column mismatch against sample_submission"
    assert len(out) == len(sample) and set(out["Run No"]) == set(sample["Run No"])
    assert out.drop(columns="Run No").isna().sum().sum() == 0, "NaNs in predictions"
    if out_path:
        out.to_csv(out_path, index=False)
        print(f"wrote {out_path} | {out.shape} | {time.time()-t0:.0f}s", flush=True)
    return out


In [7]:
t0 = time.time()
submission = run(stride=STRIDE, out_path=str(OUT_PATH))
print(f"done in {time.time() - t0:.0f}s")

  01_seasonal            +  4 cols  (9.7s)


  01g_seasonal_global    +  3 cols  (7.1s)


  09_inflow_regime       +  4 cols  (7.1s)


  02_price_relative      +  8 cols  (1.5s)


  06_spread              + 11 cols  (4.9s)


...

selected 143 count features | 297s


long: train 6,872,544 rows, 209 features | 382s


fitted 600 rounds | 2410s


wrote submission_xgboost.csv | (725, 2353) | 2457s


done in 2459s


In [8]:
# Optional: compare with the file we uploaded to Kaggle, if it is next to this notebook.
ref_path = ROOT / "submission.csv"
if ref_path.exists() and STRIDE == 1:
    ref = pd.read_csv(ref_path).set_index("Run No").loc[submission["Run No"]]
    a = submission.drop(columns="Run No").to_numpy().ravel()
    b = ref.to_numpy().ravel()
    print(f"max |difference| to the uploaded submission: {np.abs(a - b).max():.2e}")
    print(f"correlation {np.corrcoef(a, b)[0, 1]:.4f}; "
          f"same side of 0.5 in {((a > 0.5) == (b > 0.5)).mean():.1%} of cells")

max |difference| to the uploaded submission: 5.14e-01
correlation 0.9950; same side of 0.5 in 98.1% of cells
